# 01 - EDA Data Warehouse HKBP

Notebook EDA (Minggu 4): statistik deskriptif, distribusi, korelasi, outlier, dan ≥ 5 temuan tertulis.
Sumber: data warehouse SQLite hasil `python -m etl.build_warehouse`.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from etl import eda
sesi, masuk, info = eda.muat()
print(len(sesi), 'sesi |', len(masuk), 'transaksi |', info['total_huria'], 'Huria terdaftar')

984 sesi | 960 transaksi | 30 Huria terdaftar


## 1. Statistik deskriptif

In [2]:
for judul, tabel in eda.deskriptif(sesi, masuk).items():
    print(judul); display(tabel)

Kehadiran per sesi (jiwa)


,nilai
count,984
mean,291
std,139
min,127
25%,180
50%,268
75%,367
max,1.386


Pelean per sesi (Rp)


,nilai
count,984
mean,7.020.451
std,3.369.603
min,2.594.700
25%,4.347.750
50%,6.534.000
75%,9.029.625
max,31.680.000


Nominal transaksi pemasukan (Rp)


,nilai
count,960
mean,12.869.123
std,13.925.457
min,1.612.000
25%,6.200.000
50%,9.427.500
75%,16.233.749
max,223.200.000


## 2. Distribusi

In [3]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
sesi['hadir'].plot.hist(bins=30, ax=ax[0], color='#1e5a8a'); ax[0].set_title('Kehadiran per sesi')
(sesi['pelean']/1e6).plot.hist(bins=30, ax=ax[1], color='#d98e2b'); ax[1].set_title('Pelean per sesi (juta Rp)')
(sesi['pelean']/sesi['hadir']).plot.hist(bins=30, ax=ax[2], color='#2f9e6f'); ax[2].set_title('Pelean per jemaat hadir (Rp)')
plt.tight_layout(); plt.show()

## 3. Korelasi

In [4]:
corr = sesi[['hadir','pelean','jumlah_jemaat']].corr().round(3); display(corr)
plt.figure(figsize=(4.5,3.8)); plt.imshow(corr, cmap='Blues', vmin=0, vmax=1); plt.colorbar()
plt.xticks(range(3), corr.columns, rotation=30); plt.yticks(range(3), corr.columns)
for i in range(3):
    for j in range(3): plt.text(j, i, corr.iloc[i,j], ha='center', va='center')
plt.title('Korelasi Pearson'); plt.tight_layout(); plt.show()

,hadir,pelean,jumlah_jemaat
hadir,1.000,0.987,0.843
pelean,0.987,1.000,0.828
jumlah_jemaat,0.843,0.828,1.000


## 4. Outlier (IQR 1,5x per Huria dan sesi)

In [5]:
ol = eda.outlier_per_huria(sesi); print(len(ol), 'sesi outlier'); display(ol.sort_values('pelean', ascending=False).head(10))

24 sesi outlier


,huria,sesi,tanggal,pelean,median_huria
9,HKBP Medan Kota,Ibadah Siang (Bahasa Indonesia/Batak),2026-04-05,31680000,13464000
7,HKBP Kebayoran Baru,Ibadah Siang (Bahasa Indonesia/Batak),2026-04-05,29040000,12221000
8,HKBP Medan Kota,Ibadah Pagi (Bahasa Batak),2026-04-05,25920000,11016000
6,HKBP Kebayoran Baru,Ibadah Pagi (Bahasa Batak),2026-04-05,23760000,9999000
19,HKBP Sudirman Jakarta,Ibadah Siang (Bahasa Indonesia/Batak),2026-04-05,23760000,9603000
17,HKBP Rawamangun Jati,Ibadah Siang (Bahasa Indonesia/Batak),2026-04-05,21780000,9256500
21,HKBP Tebet Barat,Ibadah Siang (Bahasa Indonesia/Batak),2026-04-05,19800000,8332500
18,HKBP Sudirman Jakarta,Ibadah Pagi (Bahasa Batak),2026-04-05,19440000,7857000
23,HKBP Teladan Medan,Ibadah Siang (Bahasa Indonesia/Batak),2026-04-05,18480000,7777000
16,HKBP Rawamangun Jati,Ibadah Pagi (Bahasa Batak),2026-04-05,17820000,7573500


In [6]:
sesi.boxplot(column='pelean', by='nama_huria', rot=90, figsize=(10,4.5)); plt.suptitle(''); plt.title('Pelean per sesi menurut Huria'); plt.tight_layout(); plt.show()

## 5. Temuan awal (≥ 5)

In [7]:
for i, t in enumerate(eda.temuan(sesi, masuk, info), 1):
    print(f'{i}. {t}\n')

1. **Pelean hampir berbanding lurus dengan kehadiran** (Pearson r = 0.987); rata-rata pelean per jemaat hadir Rp 24.078 (simpangan baku Rp 1.898).

2. **Skala Huria sangat timpang**: rata-rata pelean per sesi HKBP Medan Kota (Rp 12.515.122) adalah 3.9x HKBP Percut Sei Tuan (Rp 3.207.366); analisis lintas Huria harus dinormalisasi (per jemaat / per hadir).

3. **Perbedaan antar sesi ibadah**: kehadiran rata-rata 'Ibadah Pagi (Bahasa Batak)' 262 jiwa vs 'Ibadah Siang (Bahasa Indonesia/Batak)' 321 jiwa (selisih 18%).

4. **Pemasukan memuncak pada 2026-08** sebesar 1.9x rata-rata bulanan (terendah 2026-07: 0.8x; bulan berjalan 2026-10 tidak dihitung) - pola musiman pesta/hari raya.

5. **Konsentrasi kategori**: 'Pelean Minggu' menyumbang 56% total pemasukan, diikuti 'Iuran Taon (Guguan)' (18%).

6. **Aturan alokasi stabil**: 55.0% pemasukan dialokasikan ke Setoran Pusat dan 45.0% ke Kas Huria; 96.7% alokasi pusat sudah dilunasi (sisanya adalah bulan yang belum jatuh tempo).

7. **Cakupan d